# File Metadata

## Overview

This chapter focuses on the programmatically access to file metadata. The code cells builds on the example dataset downloaded in Chapter [Data Download](https://ioerdata.training.fdz.ioer.info/notebooks/102_download_from_dataverse.html). 


>**The following step-by-step guide explains how to:**
>- 1. Extract key information from all files in the dataset,
>- 2. Inspect a GeoPackage (.gpkg),
>- 3. Explore a dataset's structure.

<hr style="border: none; border-top: 2px solid #70a401;">

## 1. Display key file metadata for each file in the dataset

File metadata describe the individual files within a dataset, including details such as file name, file size or access status (publicly accessible or restricted data). The information are available for each file on the ioerDATA website (see [](#Figure)) or can be retrieved programmatically.
 
 ```{figure} ../resources/Screenshot_File_Metadata.png
:name: Access file metadata
:label: Figure

Example: Access a specific file metadata on ioerDATA webpage.
```

Key information from file metadata are in this example displayed in two ways a) as a list, and b) as a table. 

**a) List** 

The code cell below loads all file metadata from a dataset into a local JSON file, extracts the list of files, and displays key information (file name, file size, access level (open or restricted), and available description) for each file.

In [3]:
# Extraction of the key information of file-level metadata as a list

import json

# Load metadata from local file
with open("dataset_metadata.json", "r") as f:
    dataset_metadata = json.load(f)

# Extract file list from metadata
files = dataset_metadata["data"]["latestVersion"]["files"]

# Loop through all files and display key information
for f in files:
    name = f["dataFile"]["filename"]
    size = f["dataFile"]["filesize"]
    restricted = f["restricted"]
    
    # Try to get description
    description = f.get("description")
    
    # Fallback if description is missing
    if not description:
        description = "No description available"
    
    print(f"File: {name}")
    print(f"Filesize: {size} bytes")
    print(f"Access: {'open' if not restricted else 'restricted'}")
    print(f"Explanation: {description}")
    print("-" * 40)

File: Assessment_and_Monitoring_of_Local_Climate_Regulation.pdf
Filesize: 11200912 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Climate_regulation_in_cities_as_an_ecosystem_service_German.pdf
Filesize: 16359728 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Cooling_Capacity_2018_buffered.gdb.zip
Filesize: 229243224 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Documentation.md
Filesize: 7247 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Figure_1_Urban_green_Infrastructure_Syrbe_KLu-04.png
Filesize: 471283 bytes
Access: open
Explanation: No description available
----------------------------------------
File: Map_1_Climate_regulation_Air_Photo.jpg
Filesize: 973971 bytes
Access: open
Explanation: No description available
----------------------------------------

**b) Table** 

For better readability, the key information of metadata can be displayed as a table. 

In [4]:
import pandas as pd

rows = []

for f in files:
    rows.append({
        "File": f["dataFile"]["filename"],
        "Filesize": f["dataFile"]["filesize"],
        "Access": "open" if not f["restricted"] else "restricted",
        "Explanation": f.get("description", "No description available")
    })

df = pd.DataFrame(rows)
df

,File,Filesize,Access,Explanation
0,Assessment_and_Monitoring_of_Local_Climate_Reg...,11200912,open,No description available
1,Climate_regulation_in_cities_as_an_ecosystem_s...,16359728,open,No description available
2,Cooling_Capacity_2018_buffered.gdb.zip,229243224,open,No description available
3,Documentation.md,7247,open,No description available
4,Figure_1_Urban_green_Infrastructure_Syrbe_KLu-...,471283,open,
5,Map_1_Climate_regulation_Air_Photo.jpg,973971,open,
6,Map_2_Climate_regulation_Tree_Cover.jpg,1071537,open,
7,Map_3_Climate_regulation_Population.jpg,790493,open,
8,Map_4_Climate_regulation_Cooling_Capacity.jpg,1145944,open,
9,Map_5_Cities_cooling_capacity.jpg,1697718,open,


<hr style="border: none; border-top: 2px solid #70a401;">

## Inspect a GeoPackage

The main data in this replication package are stored as a **GeoPackage (.gpkg)** and called `climate_regulation_in_cities.gpkg`. This dataset presents a national indicator of local climate regulation provided by urban green infrastructure across 165 German cities. It quantifies the physical cooling capacity of urban green spaces and the proportion (%) of the population benefiting from these services. 

In [5]:
import pandas as pd
import geopandas as gpd

from pathlib import Path

DATA_DIR = Path.cwd() / "data" / "raw"

#GPKG = "data/raw/climate_regulation_in_cities.gpkg"

# List all layers (spatial & non-spatial)
layers = gpd.list_layers(GPKG)
print("Available layers:")
print(layers)

# Filter to spatial layers (keep only those with geometry) and pick the first one by name
first_layer = layers.loc[layers.geometry_type.notna(), "name"].iloc[0]
print("Reading layer:", first_layer)  # The *first layer* is selected automatically — it may not always be the most relevant one

# Read the selected spatial layer into a GeoDataFrame
cities = gpd.read_file(GPKG, layer=first_layer) # load into a GeoDataFrame
print(cities.head()) # preview the first rows

NameError: name 'GPKG' is not defined

The **README.md file** explains the abbrevations of the variables in this spatial layer:

- **Variables:**
    - `AGS`: Official municipality key
    - `GEN`: Municipality name
    - `NUTS`: NUTS region code
    - `shape_area`: Polygon area (m²)
    - `shape_leng`: Polygon perimeter (m)
    - `Pop_Benefit_Percent`: Percentage of inhabitants favored by the cooling effect of urban green infrastructure

<hr style="border: none; border-top: 2px solid #70a401;">

## 3. Explore a dataset's structure

This code block displays a quick overview of the dataset:
- number of rows and columns
- column names
- data types (e.g., numeric, string, geometry)
- missing values

In [10]:
# Explore a dataset's structure

cities.info() # Inspect the dataset using `.info()`  
cols = [c for c in cities.columns if c != cities.geometry.name]
summary = cities[cols].describe(include="Pop_Benefit_Percent")  # Summarize key statistics using `.describe()` with
                                                                # focus on variable Pop_Benefit_Percent (excluding geometry)  
print("Key statistics") 
summary

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   AGS                  165 non-null    object  
 1   GEN                  165 non-null    object  
 2   NUTS                 165 non-null    object  
 3   Shape_Leng           165 non-null    float64 
 4   Shape_Area           165 non-null    float64 
 5   Pop_Benefit_Percent  165 non-null    float64 
 6   geometry             165 non-null    geometry
dtypes: float64(3), geometry(1), object(3)
memory usage: 9.2+ KB


,AGS,GEN,NUTS,Shape_Leng,Shape_Area,Pop_Benefit_Percent
count,165,165,165,165.000000,1.650000e+02,165.000000
unique,165,165,137,NaN,NaN,NaN
top,09563000,Fürth,DEA36,NaN,NaN,NaN
freq,1,1,6,NaN,NaN,NaN
mean,NaN,NaN,NaN,73252.695637,1.261561e+08,75.851682
std,NaN,NaN,NaN,31028.459267,1.044051e+08,10.090394
min,NaN,NaN,NaN,27046.322289,2.594714e+07,46.746620
25%,NaN,NaN,NaN,53928.592357,6.977318e+07,69.224536
50%,NaN,NaN,NaN,65442.570260,9.850075e+07,76.193042
75%,NaN,NaN,NaN,86071.639745,1.529937e+08,83.391564


<hr style="border: none; border-top: 2px solid #70a401;">